## Proyecto Final: Análisis de Cancelaciones Hoteleras

Objetivo: Identificar patrones que nos ayuden a predecir cancelaciones y optimizar la gestión del hotel.

Preparación del Entorno

In [6]:
import sqlite3
import os
import sys

# Esto le dice a Python: "Mírame en la carpeta actual"
sys.path.append(os.getcwd())

# Ahora ya puedes importar
from db_manager import crear_db_y_tabla, cargar_datos_en_db, consultar_datos
from data_loader import cargar_datos

# Crear y cargar
crear_db_y_tabla()
df = cargar_datos()
cargar_datos_en_db(df)
print("¡Todo funcionando!")

Tabla creada.
¡Éxito! Datos cargados correctamente.
Datos cargados.
¡Todo funcionando!


 Consultas SQL y Análisis

In [7]:
# Consulta SELECT 
query_select = "SELECT hotel, COUNT(*) as total FROM reservas GROUP BY hotel"
resultado = consultar_datos(query_select)
print("Resultado del SELECT:")
print(resultado)

Resultado del SELECT:
          hotel  total
0    City Hotel  79330
1  Resort Hotel  40060


In [8]:
# Consulta JOIN

# 1. Cargar el DF original
df = cargar_datos()

# 2. Tomar una muestra pequeña 
df_pequeño = df.head(500) 

# 3. Cargar esa muestra en la base de datos
cargar_datos_en_db(df_pequeño)

# 4. Ejecutar el JOIN sobre esa muestra pequeña
query_join_rapido = """
SELECT r1.hotel, r1.meal, COUNT(*) as total_reservas
FROM reservas r1
JOIN reservas r2 ON r1.hotel = r2.hotel
GROUP BY r1.hotel, r1.meal
LIMIT 5
"""
resultado_final = consultar_datos(query_join_rapido)
print("¡El JOIN funcionó con éxito!")
print(resultado_final)

¡Éxito! Datos cargados correctamente.
Datos cargados.
¡El JOIN funcionó con éxito!
          hotel meal  total_reservas
0  Resort Hotel   BB          185500
1  Resort Hotel   FB           14000
2  Resort Hotel   HB           50500


In [9]:
# Consulta INSERT

# Abrimos la conexión
conn = sqlite3.connect('mi_base_de_datos.db')
cursor = conn.cursor()

# Insertamos el nuevo registro
cursor.execute("""
    INSERT INTO reservas (hotel, meal, lead_time, arrival_date_year) 
    VALUES (?, ?, ?, ?)
""", ('City Hotel', 'HB', 10, 2026))

# Guardamos los cambios
conn.commit()

# Verificamos que se guardó
cursor.execute("SELECT * FROM reservas WHERE lead_time = 10")
fila_nueva = cursor.fetchall()

conn.close()

print("¡INSERT realizado con éxito!")
print("Datos de la fila insertada:", fila_nueva)

¡INSERT realizado con éxito!
Datos de la fila insertada: [('Resort Hotel', 0, 10, 2015, 'July', 27, 2, 0, 2, 2, 0.0, 0, 'BB', 'PRT', 'Online TA', 'TA/TO', 0, 0, 0, 'G', 'G', 0, 'No Deposit', 241.0, None, 0, 'Transient', 117.81, 0, 0, 'Check-Out', '2015-07-04'), ('Resort Hotel', 0, 10, 2015, 'July', 27, 3, 0, 2, 2, 2.0, 0, 'BB', 'USA', 'Online TA', 'TA/TO', 0, 0, 0, 'G', 'H', 0, 'No Deposit', 240.0, None, 0, 'Transient', 153.0, 1, 0, 'Check-Out', '2015-07-05'), ('Resort Hotel', 0, 10, 2015, 'July', 28, 10, 0, 1, 2, 0.0, 0, 'BB', 'USA', 'Direct', 'Direct', 0, 0, 0, 'A', 'A', 0, 'No Deposit', 250.0, None, 0, 'Transient', 131.0, 0, 0, 'Check-Out', '2015-07-11'), ('Resort Hotel', 0, 10, 2015, 'July', 29, 16, 0, 3, 2, 0.0, 0, 'BB', 'ESP', 'Direct', 'Direct', 0, 0, 0, 'D', 'D', 0, 'No Deposit', 250.0, None, 0, 'Transient', 184.0, 1, 0, 'Check-Out', '2015-07-19'), ('Resort Hotel', 1, 10, 2015, 'July', 29, 16, 0, 3, 2, 0.0, 0, 'BB', 'PRT', 'Direct', 'Direct', 0, 0, 0, 'A', 'A', 0, 'No Deposit',

In [10]:
# celda de consulta
resultado = consultar_datos("SELECT * FROM reservas LIMIT 5")
print(resultado)

          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  deposit_type  \
0                        0                     0       2  ...    No Deposit   
1     